In [3]:
import pandas as pd

customers = pd.read_csv("olist_customers_dataset.csv")
geolocation = pd.read_csv("olist_geolocation_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
order_payments = pd.read_csv("olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("olist_order_reviews_dataset.csv")
orders = pd.read_csv("olist_orders_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")
category_translation = pd.read_csv("product_category_name_translation.csv")

In [24]:
customers_clean = customers.copy()
geolocation_clean = geolocation.copy()
order_items_clean = order_items.copy()
order_payments_clean = order_payments.copy()
order_reviews_clean = order_reviews.copy()
orders_clean = orders.copy()
products_clean = products.copy()
sellers_clean = sellers.copy()
category_translation_clean = category_translation.copy()

In [5]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders_clean[col] = pd.to_datetime(
        orders_clean[col],
        errors='coerce'
    )

orders_clean[date_columns].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [6]:
orders_clean[date_columns].isna().sum()

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [7]:
print(orders_clean['order_status'].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [8]:
print(
    orders_clean.loc[
        orders_clean['order_status'] == 'delivered',
        'order_delivered_customer_date'
    ].isna().sum()
)

8


In [9]:
print(
    orders_clean.loc[
        orders_clean['order_delivered_customer_date'].notna(),
        'order_status'
    ].value_counts()
)

order_status
delivered    96470
canceled         6
Name: count, dtype: int64


In [12]:
anomalies = orders_clean[
    (
        (orders_clean['order_status'] == 'delivered') &
        (orders_clean['order_delivered_customer_date'].isna())
    )
    |
    (
        (orders_clean['order_status'] != 'delivered') &
        (orders_clean['order_delivered_customer_date'].notna())
    )
]
anomalies[[
    'order_id',
    'order_status',
    'order_purchase_timestamp',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]]

,order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,canceled,2018-02-19 19:48:52,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-30 18:12:23,NaT,2017-12-18
8791,dabf2b0e35b423f94618bf965fcb7514,canceled,2016-10-09 00:56:52,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-03 13:57:00,NaT,2018-07-30
58266,770d331c84e5b214bd9dc70a10b829d0,canceled,2016-10-07 14:52:30,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29
59332,8beb59392e21af5eb9547ae1a9938d06,canceled,2016-10-08 20:17:50,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-03 09:28:00,NaT,2018-07-24
92636,65d1e226dfaeb8cdc42f665422522d14,canceled,2016-10-03 21:01:41,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25


In [13]:
orders_clean['order_id'].duplicated().sum()

np.int64(0)

In [14]:
orders_clean['order_id'].nunique()

99441

In [15]:
len(orders_clean)

99441

In [16]:
orders_clean['order_status'].unique()

array(['delivered', 'invoiced', 'shipped', 'processing', 'unavailable',
       'canceled', 'created', 'approved'], dtype=object)

In [17]:
orders_clean['order_status'].isna().sum()

np.int64(0)

In [18]:
# Purchase should not happen after approval
print(
    (orders_clean['order_purchase_timestamp'] >
     orders_clean['order_approved_at']).sum()
)

0


In [19]:
# Approval should not happen after carrier handoff
print(
    (orders_clean['order_approved_at'] >
     orders_clean['order_delivered_carrier_date']).sum()
)

1359


In [20]:
# Carrier handoff should not happen after customer delivery
print(
    (orders_clean['order_delivered_carrier_date'] >
     orders_clean['order_delivered_customer_date']).sum()
)

23


In [21]:
date_anomalies = orders_clean[
    (
        orders_clean['order_approved_at'].notna() &
        orders_clean['order_delivered_carrier_date'].notna() &
        (orders_clean['order_approved_at'] >
         orders_clean['order_delivered_carrier_date'])
    )
    |
    (
        orders_clean['order_delivered_carrier_date'].notna() &
        orders_clean['order_delivered_customer_date'].notna() &
        (orders_clean['order_delivered_carrier_date'] >
         orders_clean['order_delivered_customer_date'])
    )
]

date_anomalies[[
    'order_id',
    'order_status',
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]].head(20)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
15,dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04
64,688052146432ef8253587b930b01a06d,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15
199,58d4c4747ee059eeeb865b349b41f53a,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31
210,412fccb2b44a99b36714bca3fef8ad7b,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31
415,56a4ac10a4a8f2ba7693523bb439eede,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06
481,32e4fa9bb468884309b58b9348de70c3,delivered,2018-07-04 16:49:21,2018-07-05 16:33:06,2018-07-05 14:50:00,2018-07-07 14:41:18,2018-07-23
483,4df92d82d79c3b52c7138679fa9b07fc,delivered,2018-07-24 11:32:11,2018-07-29 23:30:52,2018-07-26 14:46:00,2018-07-27 18:55:57,2018-08-06
585,16e38caa92e342c7780f68832f832d4d,delivered,2018-05-07 01:09:09,2018-05-07 16:52:39,2018-05-07 15:09:00,2018-05-24 00:31:18,2018-06-07
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37,2018-09-04
817,6051e6d3da9a50b7325cbe9c81025062,delivered,2018-07-03 23:40:16,2018-07-05 16:31:26,2018-07-04 12:14:00,2018-07-05 22:52:28,2018-07-19


In [22]:
# 1. Overall scale
print(f"Total anomalies: {len(date_anomalies)}")
print(date_anomalies['order_status'].value_counts())

# 2. Split the two anomaly types
approved_after_carrier = orders_clean[
    orders_clean['order_approved_at'].notna() &
    orders_clean['order_delivered_carrier_date'].notna() &
    (orders_clean['order_approved_at'] > orders_clean['order_delivered_carrier_date'])
].copy()

carrier_after_customer = orders_clean[
    orders_clean['order_delivered_carrier_date'].notna() &
    orders_clean['order_delivered_customer_date'].notna() &
    (orders_clean['order_delivered_carrier_date'] > orders_clean['order_delivered_customer_date'])
].copy()

print(f"'approved after carrier' anomalies: {len(approved_after_carrier)}")
print(f"'carrier after customer' anomalies: {len(carrier_after_customer)}")

# 3. Measure size of the gap (in hours)
approved_after_carrier['approval_lag_hours'] = (
    approved_after_carrier['order_approved_at'] - 
    approved_after_carrier['order_delivered_carrier_date']
).dt.total_seconds() / 3600

print(approved_after_carrier['approval_lag_hours'].describe())

# 4. Flag rows instead of dropping
orders_clean['has_date_anomaly'] = orders_clean.index.isin(date_anomalies.index)

# quick sanity check
print(orders_clean['has_date_anomaly'].sum(), "flagged rows out of", len(orders_clean))

Total anomalies: 1382
order_status
delivered    1373
shipped         9
Name: count, dtype: int64
'approved after carrier' anomalies: 1359
'carrier after customer' anomalies: 23
count    1359.000000
mean       24.751987
std       115.307793
min         0.005833
25%         1.415417
50%        17.167778
75%        25.956667
max      4109.256111
Name: approval_lag_hours, dtype: float64
1382 flagged rows out of 99441


In [23]:
# Recompute lag for all approved_after_carrier rows (in case not already done)
approved_after_carrier['approval_lag_hours'] = (
    approved_after_carrier['order_approved_at'] - 
    approved_after_carrier['order_delivered_carrier_date']
).dt.total_seconds() / 3600

# Split into minor lag vs severe anomaly
minor_lag_ids = approved_after_carrier[approved_after_carrier['approval_lag_hours'] <= 48]['order_id']
severe_lag_ids = approved_after_carrier[approved_after_carrier['approval_lag_hours'] > 48]['order_id']

print(f"Minor lag (<=48h): {len(minor_lag_ids)}")
print(f"Severe anomaly (>48h): {len(severe_lag_ids)}")

# Build a single data_quality_flag column
orders_clean['data_quality_flag'] = 'ok'
orders_clean.loc[orders_clean['order_id'].isin(minor_lag_ids), 'data_quality_flag'] = 'minor_lag'
orders_clean.loc[orders_clean['order_id'].isin(severe_lag_ids), 'data_quality_flag'] = 'severe_anomaly'
orders_clean.loc[orders_clean['order_id'].isin(carrier_after_customer['order_id']), 'data_quality_flag'] = 'severe_anomaly'

print(orders_clean['data_quality_flag'].value_counts())

# For time-based metrics later, exclude severe_anomaly rows:
orders_for_timing = orders_clean[orders_clean['data_quality_flag'] != 'severe_anomaly'].copy()
print(f"Rows usable for timing analysis: {len(orders_for_timing)} out of {len(orders_clean)}")

Minor lag (<=48h): 1181
Severe anomaly (>48h): 178
data_quality_flag
ok                98059
minor_lag          1181
severe_anomaly      201
Name: count, dtype: int64
Rows usable for timing analysis: 99240 out of 99441


In [25]:
orders_clean.to_csv('orders_clean.csv', index=False)